# EXP-002 BM25 메타데이터 dev-small 확인

목적: BM25 문서에 메타데이터를 붙인 hybrid(EXP-002)가 동작하고 EXP-001 대비 방향을 dev-small에서 본다. 판정은 dev-full·multi-doc으로만 한다(`docs/experiments/EXP-002.md`). dev-small 점수는 dev-full과 비교하지 않는다. dev-small은 single-doc 질의뿐이라 이 가설의 주 효과(multi-doc)는 여기서 보이지 않는다.

In [1]:
from pathlib import Path

import pandas as pd

from rag.eval.metrics import gold_ranks, load_cfg, load_gold, min_docs, per_query, read_run

In [2]:
ROOT = Path.cwd().parent
LAYER = "dev-small"
RUNS = {
    "exp001": ROOT / "data/runs/kure-v1-fixed-512-64-hybrid-dev-small.csv",
    "exp002": ROOT / "data/runs/kure-v1-fixed-512-64-hybrid-meta-dev-small.csv",
}

## 순위 파일

순위 파일은 `uv run python -m rag.search +exp=exp002`로 만들었다. 질의마다 서로 다른 문서 10개가 있는지 본다.

In [3]:
runs = {name: read_run(path) for name, path in RUNS.items()}
for name, run in runs.items():
    per_q = run.groupby("qid")["doc_id"].nunique()
    print(name, run.shape, "질의", per_q.size, "질의당 문서 수", per_q.min(), "~", per_q.max())

exp001 (10390, 3) 질의 1039 질의당 문서 수 10 ~ 10
exp002 (10390, 3) 질의 1039 질의당 문서 수 10 ~ 10


## 지표 비교

In [4]:
cfg = load_cfg()
ev = cfg.eval
gold = load_gold(cfg, LAYER)
n_min = min_docs(ev.recall_ks, ev.mrr_k)
ranks = {name: gold_ranks(run, gold, n_min) for name, run in runs.items()}
scores = {name: per_query(r, ev.recall_ks, ev.mrr_k) for name, r in ranks.items()}
pd.DataFrame({name: s.mean() for name, s in scores.items()}).round(4)

,exp001,exp002
recall@1,0.9105,0.8816
recall@5,0.9808,0.9711
recall@10,0.9875,0.9808
mrr@10,0.9438,0.9228


In [5]:
pd.DataFrame({
    name: s.join(gold["meeting_name"]).groupby("meeting_name")["recall@5"].mean()
    for name, s in scores.items()
}).assign(n=gold["meeting_name"].value_counts()).round(4)

,exp001,exp002,n
meeting_name,,,
국정감사,0.9913,0.9855,345
본회의,0.9789,0.9684,95
소위원회,0.9726,0.9566,438
예산결산특별위원회,0.9643,0.9643,56
특별위원회,0.9905,0.9905,105


## 상위 5위 진입·이탈

EXP-002가 EXP-001의 5위 안 정답을 밀어낸 질의(이탈)와 5위 밖 정답을 끌어올린 질의(진입)를 센다.

In [6]:
hit = pd.DataFrame({name: s["recall@5"] for name, s in scores.items()})
pd.crosstab(hit["exp001"], hit["exp002"], rownames=["exp001@5"], colnames=["exp002@5"])

exp002@5,0.0,1.0
exp001@5,,
0.0,18,2
1.0,12,1007
